# X02 In-context Learning 与 Induction Heads


## 目标

把 induction 作为具体电路对象来研究，是理解 transformer 为什么会复制模式和形成 in-context behavior 的经典入口。


## 为什么现在学这篇

它把 attention 读图、circuit 语言和具体行为现象连起来，是进入经典 transformer-circuits 的第一站。


## Notebook 与交付

- 原文: https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html
- Notebook：`notebooks/extensions/zh/x02_induction_heads.ipynb`
- 先修: X01, M06
- 在 Colab 里复现一个最小 copying task，并说明 induction head 为什么比单个 attention 热点更像“机制”。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import math
import matplotlib.pyplot as plt
import numpy as np

tokens = ["A", "B", "C", "A", "B", "?"]
prev_tokens = ["<bos>", "A", "B", "C", "A", "B"]
vocab = {"<bos>": [0, 0, 0], "A": [1, 0, 0], "B": [0, 1, 0], "C": [0, 0, 1]}
keys = np.array([vocab[token] for token in prev_tokens[:-1]], dtype=float)
values = np.array([vocab[token] for token in tokens[:-1]], dtype=float)
query = np.array(vocab[prev_tokens[-1]], dtype=float)

scores = 5.0 * (keys @ query) / math.sqrt(query.shape[0])
weights = np.exp(scores - scores.max())
weights = weights / weights.sum()
prediction = weights @ values
winner = ["A", "B", "C"][int(np.argmax(prediction))]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(range(len(weights)), weights, color="#1f5f8b")
ax.set_xticks(range(len(weights)), [f"pos {i}:{token}" for i, token in enumerate(tokens[:-1])], rotation=20)
ax.set_ylabel("attention weight")
ax.set_title("Final query attends to earlier positions")
plt.tight_layout()

print("Final previous-token query:", prev_tokens[-1])
print("Predicted next-token vector:", np.round(prediction, 3))
print("Recovered token:", winner)


## 小结

Induction 不只是“注意力亮了一下”，而是一个依赖重复上下文的复制机制。


## Colab 优先的复现流程


### 运行前

- 在运行前先写 1 条你对机制的预测。
- 先写清你要对比的 baseline 是什么。
- 先决定什么结果会推翻你最喜欢的解释。


### 运行后

- 在笔记里把 observation 和 inference 分开。
- 标出复现之后仍然存在的 1 个歧义。
- 写 1 个能降低该歧义的下一步实验。


### 最后交付这些产物

- 在 Colab 里复现一个最小 copying task，并说明 induction head 为什么比单个 attention 热点更像“机制”。
- 1 份 experiment log，写清你改了哪些设置。
- 1 段“这次复现仍然不能证明什么”。


## 验收题

- 为什么“某个位置注意到了前文”还不等于“发现了 induction 机制”？
- 你的 toy copying 任务里，重复 token 距离变化时性能怎么变？
- 如果没有前序 head 提供匹配线索，induction head 会失去什么？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就回去重跑复现并重写 memo。
